## 1. Install dependencies

In [8]:
!pip install -q gradio huggingface_hub transformers torch torchvision pillow

## 2. Load the clothes segmentation model

In [1]:
import torch
import numpy as np
from PIL import Image, ImageFilter
from transformers import SegformerImageProcessor, AutoModelForSemanticSegmentation

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.73k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/109M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

In [ ]:
processor = SegformerImageProcessor.from_pretrained("mattmdjaga/segformer_b2_clothes")
seg_model = AutoModelForSemanticSegmentation.from_pretrained("mattmdjaga/segformer_b2_clothes")
seg_model.eval().to(device)

In [15]:
UPPER_LABELS = [4, 7]   # Upper-clothes, Dress
LOWER_LABELS = [5, 6]   # Skirt, Pants

#   0  = Background
#   1  = Hat
#   2  = Hair
#   3  = Sunglasses
#   4  = Upper-clothes   ← UPPER selection
#   5  = Skirt           ← LOWER selection
#   6  = Pants           ← LOWER selection
#   7  = Dress           ← UPPER selection
#   8  = Belt
#   9  = Left-shoe
#   10 = Right-shoe
#   11 = Face
#   12 = Left-leg
#   13 = Right-leg
#   14 = Left-arm
#   15 = Right-arm
#   16 = Bag
#   17 = Scarf

In [ ]:
def get_clothing_mask(image: Image.Image, body_part: str) -> Image.Image:
    """Returns a black/white mask (white = the segmented clothing region)."""
    image = image.convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = seg_model(**inputs)

    logits = outputs.logits  # shape: (1, num_labels, H, W)
    upsampled = torch.nn.functional.interpolate(
        logits, size=image.size[::-1], mode="bilinear", align_corners=False
    )
    pred_seg = upsampled.argmax(dim=1)[0].cpu()  # (H, W) class id per pixel

    labels = UPPER_LABELS if body_part == "Upper Body" else LOWER_LABELS

    mask = np.zeros(pred_seg.shape, dtype=np.uint8)
    for label_id in labels:
        mask[pred_seg.numpy() == label_id] = 255

    mask_img = Image.fromarray(mask).convert("L")
    mask_img = mask_img.filter(ImageFilter.MaxFilter(9))  # slightly grow edges
    return mask_img

## 3. Inpainting via the external AI API (Hugging Face Inference Providers)

In [ ]:
from huggingface_hub import InferenceClient
from io import BytesIO

HF_TOKEN = "*****************"  # https://huggingface.co/settings/tokens

hf_client = InferenceClient(provider="fal-ai", api_key=HF_TOKEN)

def build_edit_instruction(body_part: str, prompt: str) -> str:
    region = "upper body clothing (shirt/jacket/top)" if body_part == "Upper Body" else "lower body clothing (pants/skirt)"
    return (
        f"Change only the {region} to: {prompt}. "
        f"Keep the person's face, pose, body shape, and background exactly the same."
    )

def inpaint_clothing(image: Image.Image, body_part: str, prompt: str) -> Image.Image:
    buffer = BytesIO()
    image.convert("RGB").save(buffer, format="PNG")
    image_bytes = buffer.getvalue()

    instruction = build_edit_instruction(body_part, prompt)

    result = hf_client.image_to_image(
        image_bytes,
        prompt=instruction,
        model="black-forest-labs/FLUX.1-Kontext-dev",
    )
    return result  # PIL.Image


## 4. Full pipeline

Upload image → Segment chosen body part (for visual verification) → Call the external API → Return result.


In [3]:
def virtual_try_on(image: Image.Image, body_part: str, prompt: str):
    if image is None:
        return None, None, "Please upload an image first."
    if not prompt or prompt.strip() == "":
        prompt = "stylish new clothing, high quality, realistic fabric"

    try:
        image = image.convert("RGB")
        image.thumbnail((768, 768))

        # Step 1: segment the chosen region (shown in the UI as proof of segmentation)
        mask = get_clothing_mask(image, body_part)

        # Step 2: inpaint using the external AI API
        result = inpaint_clothing(image, body_part, prompt)

        return result, mask, "Done via Hugging Face API ✅"
    except Exception as e:
        return None, None, f"API Error: {e}"

## 5. Gradio UI

In [8]:
import gradio as gr

with gr.Blocks(title="Virtual Try-On") as demo:
    gr.Markdown("# 👕 Virtual Try-On System")
    gr.Markdown("Upload a photo, choose Upper or Lower body, describe the new clothing, then click **Try On**.")

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(type="pil", label="Upload Image")
            body_part = gr.Radio(["Upper Body", "Lower Body"], label="Select Body Part", value="Upper Body")
            prompt = gr.Textbox(label="Describe new clothing (e.g. 'black leather jacket', 'blue denim jeans')")
            btn = gr.Button("Try On", variant="primary")
        with gr.Column():
            output_image = gr.Image(type="pil", label="Result")
            mask_preview = gr.Image(type="pil", label="Segmentation Mask (for reference)")
            status = gr.Textbox(label="Status", interactive=False)

    btn.click(fn=virtual_try_on, inputs=[input_image, body_part, prompt], outputs=[output_image, mask_preview, status])

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://39ccc196f82e7e2b29.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://39ccc196f82e7e2b29.gradio.live


<table>
  <tr>
    <td><img src="https://i.postimg.cc/wxX0JkcF/man.webp" width="200"><br><center>Main Image</center></td>
    <td><img src="https://i.postimg.cc/T1MNx7js/Screenshot-from-2026-07-16-16-52-52.png" width="200"><br><center>Mask 1(red denim jeans) </center></td>
    <td><img src="https://i.postimg.cc/pLG0rXY9/Screenshot-from-2026-07-16-16-52-16.png" width="200"><br><center>Mask 2(black leather jacket)</center></td>
  </tr>
</table>